<a href="https://colab.research.google.com/github/Raditya-Z/MachineLearning_244107020144/blob/main/JS-11/JS_11_TugasPraktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📶 Tugas Praktikum

# 1. Klasifikasi data voice.csv

A. Import library dan membaca data

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv("voice.csv")

print("Ukuran dataset:", df.shape)
print(df.head())
print("Kolom:", df.columns.tolist())

Ukuran dataset: (3168, 21)
   meanfreq        sd    median       Q25       Q75       IQR       skew  \
0  0.059781  0.064241  0.032027  0.015071  0.090193  0.075122  12.863462   
1  0.066009  0.067310  0.040229  0.019414  0.092666  0.073252  22.423285   
2  0.077316  0.083829  0.036718  0.008701  0.131908  0.123207  30.757155   
3  0.151228  0.072111  0.158011  0.096582  0.207955  0.111374   1.232831   
4  0.135120  0.079146  0.124656  0.078720  0.206045  0.127325   1.101174   

          kurt    sp.ent       sfm  ...  centroid   meanfun    minfun  \
0   274.402906  0.893369  0.491918  ...  0.059781  0.084279  0.015702   
1   634.613855  0.892193  0.513724  ...  0.066009  0.107937  0.015826   
2  1024.927705  0.846389  0.478905  ...  0.077316  0.098706  0.015656   
3     4.177296  0.963322  0.727232  ...  0.151228  0.088965  0.017798   
4     4.333713  0.971955  0.783568  ...  0.135120  0.106398  0.016931   

     maxfun   meandom    mindom    maxdom   dfrange   modindx  label  
0  0.2

B. Menyiapkan fitur dan label

In [10]:
# Kolom target pada dataset voice.csv
target_col = "label"

X = df.drop(columns=[target_col])
y = df[target_col]

print("Jumlah fitur:", X.shape[1])
print("Distribusi label:")
print(y.value_counts())

Jumlah fitur: 20
Distribusi label:
label
male      1584
female    1584
Name: count, dtype: int64


C. Melatih enam kombinasi SVM

In [11]:
hasil = []

splits = {
    "70:30": 0.30,
    "80:20": 0.20
}

kernels = {
    "Linear": SVC(kernel="linear"),
    "Polynomial": SVC(kernel="poly"),
    "RBF": SVC(kernel="rbf")
}

for nama_split, test_size in splits.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    for nama_kernel, svm in kernels.items():

        model = make_pipeline(
            StandardScaler(),
            svm
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        akurasi = accuracy_score(y_test, y_pred)

        hasil.append({
            "Rasio Split": nama_split,
            "Kernel": nama_kernel,
            "Akurasi": akurasi,
            "Akurasi (%)": round(akurasi * 100, 2)
        })

        print(
            f"Split {nama_split} | "
            f"Kernel {nama_kernel} | "
            f"Akurasi: {akurasi:.4f}"
        )

tabel_hasil = pd.DataFrame(hasil)

print("\nTabel Perbandingan Akurasi")
display(
    tabel_hasil[
        ["Rasio Split", "Kernel", "Akurasi (%)"]
    ]
)

Split 70:30 | Kernel Linear | Akurasi: 0.9790
Split 70:30 | Kernel Polynomial | Akurasi: 0.9600
Split 70:30 | Kernel RBF | Akurasi: 0.9832
Split 80:20 | Kernel Linear | Akurasi: 0.9748
Split 80:20 | Kernel Polynomial | Akurasi: 0.9558
Split 80:20 | Kernel RBF | Akurasi: 0.9826

Tabel Perbandingan Akurasi


,Rasio Split,Kernel,Akurasi (%)
0,70:30,Linear,97.90
1,70:30,Polynomial,96.00
2,70:30,RBF,98.32
3,80:20,Linear,97.48
4,80:20,Polynomial,95.58
5,80:20,RBF,98.26


# 2. Klasifikasi citra siang dan malam

A. Import library dan memuat gambar

In [12]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Ambil seluruh gambar dari folder dataset
image_dir = Path("/content/drive/MyDrive/Machine Learning 2026/images/images/")

image_paths = list(image_dir.rglob("*.jpg"))
image_paths += list(image_dir.rglob("*.jpeg"))
image_paths += list(image_dir.rglob("*.png"))

print("Jumlah gambar:", len(image_paths))

Jumlah gambar: 400


B. Ekstraksi fitur histogram

Setiap gambar diubah ukurannya terlebih dahulu, kemudian histogram setiap kanal warna dihitung menggunakan 8 bin per kanal. Total terdapat 24 fitur untuk setiap gambar.

In [13]:
def extract_histogram(image_path, bins=8):
    image = cv2.imread(str(image_path))

    if image is None:
        return None

    image = cv2.resize(image, (1100, 600))

    features = []

    for channel in range(3):
        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [bins],
            [0, 256]
        )

        hist = cv2.normalize(
            hist, None, alpha=1, beta=0,
            norm_type=cv2.NORM_L1
        )

        features.extend(hist.flatten())

    return np.array(features, dtype=np.float32)


X = []
y = []

for path in image_paths:
    label = path.parent.name.lower()

    if label not in ["day", "night"]:
        continue

    features = extract_histogram(path)

    if features is not None:
        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Shape fitur:", X.shape)
print("Shape label:", y.shape)
print("Distribusi kelas:")
print(pd.Series(y).value_counts())

Shape fitur: (400, 24)
Shape label: (400,)
Distribusi kelas:
day      200
night    200
Name: count, dtype: int64


C. Membagi data dengan rasio 80:20

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)

Data training: (320, 24)
Data testing: (80, 24)


D. Membuat model SVM dengan kernel RBF

In [15]:
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf")
)

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.001, 0.01, 0.1]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Parameter terbaik:", grid_search.best_params_)
print("Akurasi validasi terbaik:",
      round(grid_search.best_score_ * 100, 2), "%")

Parameter terbaik: {'svc__C': 1, 'svc__gamma': 0.1}
Akurasi validasi terbaik: 99.69 %


E. Evaluasi model terbaik

In [16]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

akurasi_test = accuracy_score(y_test, y_pred)

print("Akurasi testing:",
      round(akurasi_test * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Akurasi testing: 97.5 %

Classification Report:
              precision    recall  f1-score   support

         day       0.97      0.97      0.97        40
       night       0.97      0.97      0.97        40

    accuracy                           0.97        80
   macro avg       0.97      0.97      0.97        80
weighted avg       0.97      0.97      0.97        80



**Analisis**

> Berdasarkan hasil praktikum, model klasifikasi gambar siang dan malam menggunakan metode Support Vector Machine (SVM) dengan kernel RBF mendapatkan akurasi testing sebesar 97,5%. Artinya, dari 80 gambar yang diuji, sebanyak 78 gambar berhasil diklasifikasikan dengan benar, sedangkan 2 gambar lainnya salah diklasifikasikan.

> Pada proses pengujian, fitur yang digunakan adalah histogram warna. Histogram ini membantu komputer mengenali pola warna pada gambar untuk membedakan kondisi siang dan malam. Model SVM menggunakan kernel RBF dipilih karena dapat membantu memisahkan data berdasarkan pola yang dipelajari.

> Berdasarkan proses pencarian hyperparameter, diperoleh nilai terbaik yaitu C = 1 dan gamma = 0.1, dengan akurasi validasi sebesar 99,69%. Sementara itu, akurasi pada data testing sebesar 97,5%. Perbedaan ini menunjukkan bahwa hasil validasi dan hasil pengujian pada data yang berbeda tidak selalu sama.

> Secara keseluruhan, hasil tersebut menunjukkan bahwa model mampu membedakan gambar siang dan malam dengan sangat baik pada dataset yang digunakan. Meskipun demikian, masih terdapat dua gambar yang salah diprediksi sehingga model belum sepenuhnya sempurna.

**Kesimpulan**
> Berdasarkan praktikum yang telah dilakukan, dapat disimpulkan bahwa metode SVM dengan kernel RBF dan fitur histogram warna dapat digunakan untuk mengklasifikasikan gambar siang dan malam. Dengan pembagian data 80% untuk training dan 20% untuk testing, model memperoleh akurasi testing sebesar 97,5%.

> Hasil ini menunjukkan bahwa model memiliki kemampuan yang baik dalam membedakan gambar siang dan malam. Namun, masih ada kemungkinan kesalahan klasifikasi karena pola warna pada beberapa gambar dapat terlihat mirip. Oleh karena itu, penggunaan fitur histogram warna dan metode SVM RBF sudah memberikan hasil yang baik pada percobaan ini, tetapi performanya tetap perlu diuji menggunakan dataset yang lebih beragam.